# z028 - Optuna con target juntar (BAJA+1 + BAJA+2), receta del submit 32

**Por qué:** los hiperparámetros del 32 salen de z005, que buscó con target **solo BAJA+2**. Después se adoptó juntar
(z009, z019, z024, z026) sin volver a buscar. Acá se repite z005 tal cual, cambiando **solo el target**.

1. **Optuna, 40 trials** (como z005, pero **cada trial promedia 5 semillas**: las del submit 15, distintas de las de la revalidación), train 202104 → valid 202106, 761 variables, `num_leaves` 8-256 (log),
   `min_sum_hessian_in_leaf` 0,001-10 (log), árboles por early stopping con AUC (paciencia 200, tope 3000), ganancia máx con BAJA+2.
   SQLite en `C:/Users/keila/optuna/z028.db` (retoma si se corta).
   Por qué 5 semillas: con 1 semilla el ruido de un trial es ±15 M (z005: el 1° de Optuna era suerte); con 5 baja a ~±7 M,
   así Optuna ordena y se guía mejor y es menos probable que un trial bueno quede afuera del top 10 por mala semilla.
   Valor del trial = promedio de las 5 ganancias máx; árboles = promedio de los 5 early stopping.
2. **Revalidación:** los 10 mejores con **10 semillas** (las de z024-z027) en **junio** (04→06) y **mayo** (03→05).
   Referencia = trial 34 de z005 con juntar (predicciones de `probs_z026/ref_*`, mismas semillas).

**Regla (definida antes de correr, 11/10):** un trial nuevo **pasa** si contra la referencia, en **los dos cortes**, gana en ≥ 8/10 semillas
(ganancia máx) **y** mejora la ganancia media en N fijo (7,29% del mes). Si pasan **3 o más**, el ensamble nuevo = los que pasan
(hasta 10, ordenados por el promedio de los dos cortes). Si pasan menos de 3, el 32 queda como está.
Ojo: el early stopping y el ranking de Optuna miran junio → mayo es el control contra el sobreajuste (lección de z012).

In [1]:
# --- 1. Librerías y parámetros ---
import os, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO   = 27_500
SEMILLA   = 501419                 # la de z005 (sampler de Optuna)
SEMILLAS_OPTUNA = [501419, 623353, 786349, 834569, 946331]   # cada trial promedia 5 modelos (semillas del submit 15)
SEMILLAS  = [270131, 417293, 583117, 692347, 851071,
             135791, 246809, 314159, 527183, 964201]    # revalidación (las de la referencia en probs_z026)
CORTES    = {"junio": (202104, 202106), "mayo": (202103, 202105)}
PROP_N    = 12_000 / 164_647
N_TRIALS, TOP_REVALIDAR, MAX_ARBOLES = 40, 10, 3000
STORAGE   = "sqlite:///C:/Users/keila/optuna/z028.db"   # fuera de OneDrive
ESTUDIO   = "z028_juntar_5sem_s501419"   # nombre nuevo (el de 1 semilla quedó a medias)
CARPETA, CARPETA_REF = "probs_z028", "probs_z026"
os.makedirs(CARPETA, exist_ok=True)

params_fijos = {
    "objective": "binary", "metric": "auc", "learning_rate": 0.005, "min_data_in_leaf": 0,
    "feature_fraction": 0.5, "feature_fraction_bynode": 0.2, "max_bin": 31, "verbose": -1,
}

def log(texto):
    with open("z028_progreso.txt", "a", encoding="utf-8") as f:
        f.write(time.strftime("%H:%M:%S ") + texto + "\n")
log("inicio")

In [2]:
# --- 2. Datos (202103-202106), target juntar y lags/deltas 1 y 2 (las 761 del submit 32) ---
df = pd.read_parquet("../competencia_01.parquet")
df = df[df["foto_mes"] <= 202106]
df = df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)
df["y_juntar"] = df["clase_ternaria"].isin(["BAJA+1", "BAJA+2"]).astype(int)
no_usar = ["foto_mes", "clase_ternaria", "y_juntar"]
cols = [c for c in df.columns if c not in no_usar and c != "numero_de_cliente"]
grupo = df.groupby("numero_de_cliente")[cols]
partes = []
for k in [1, 2]:
    lag = grupo.shift(k).astype("float32")
    delta = (df[cols] - lag).astype("float32")
    lag.columns   = [f"{c}_lag{k}" for c in cols]
    delta.columns = [f"{c}_delta{k}" for c in cols]
    partes += [lag, delta]
df = pd.concat([df] + partes, axis=1)
del partes, lag, delta, grupo
variables = [c for c in df.columns if c not in no_usar]
print("Variables:", len(variables))
foto = df["foto_mes"].values

def datos_corte(corte):
    mes_train, mes_valid = CORTES[corte]
    tr, va = foto == mes_train, foto == mes_valid
    gan = np.where(df.loc[va, "clase_ternaria"].values == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO)   # ganancia con BAJA+2
    return tr, va, gan

def curva(prob, gan):
    return np.cumsum(gan[np.argsort(-prob)])

Variables: 761


In [3]:
# --- 3. Optuna (como z005, target juntar, 5 semillas por trial) ---
tr, va, gan_junio = datos_corte("junio")
datos_train = lgb.Dataset(df.loc[tr, variables], label=df.loc[tr, "y_juntar"], free_raw_data=False)
datos_valid = lgb.Dataset(df.loc[va, variables], label=df.loc[va, "y_juntar"], reference=datos_train, free_raw_data=False)
X_valid = df.loc[va, variables]

def objetivo(trial):
    inicio = time.time()
    p = {**params_fijos,
         "num_leaves": trial.suggest_int("num_leaves", 8, 256, log=True),
         "min_sum_hessian_in_leaf": trial.suggest_float("min_sum_hessian_in_leaf", 1e-3, 10, log=True),
         }
    gan_max, arboles, mejor_n = [], [], []
    for s in SEMILLAS_OPTUNA:                         # mismo trial, 5 semillas -> se promedia (menos ruido)
        modelo = lgb.train({**p, "seed": s}, datos_train, num_boost_round=MAX_ARBOLES, valid_sets=[datos_valid],
                           callbacks=[lgb.early_stopping(200, verbose=False)])
        best_iter = modelo.best_iteration or modelo.num_trees()
        acum = curva(modelo.predict(X_valid, num_iteration=best_iter), gan_junio)
        gan_max.append(acum.max() / 1e6); arboles.append(best_iter); mejor_n.append(int(acum.argmax()) + 1)
    trial.set_user_attr("best_iter", int(round(np.mean(arboles))))      # árboles para la revalidación y el submit
    trial.set_user_attr("mejor_n", int(round(np.mean(mejor_n))))
    trial.set_user_attr("desvio_M", float(np.std(gan_max)))
    log(f"trial {trial.number}: {np.mean(gan_max):.1f} ± {np.std(gan_max):.1f} M, hojas {p['num_leaves']}, "
        f"árboles {int(np.mean(arboles))}, {time.time() - inicio:.0f} s")
    return float(np.mean(gan_max))

optuna.logging.set_verbosity(optuna.logging.WARNING)
estudio = optuna.create_study(study_name=ESTUDIO, storage=STORAGE, load_if_exists=True,
                              direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEMILLA))
hechos = len([t for t in estudio.trials if t.state == optuna.trial.TrialState.COMPLETE])
estudio.optimize(objetivo, n_trials=max(0, N_TRIALS - hechos))
del datos_train, datos_valid, X_valid

trials = estudio.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
trials = trials[trials["state"] == "COMPLETE"].sort_values("value", ascending=False)
trials.to_csv("z028_optuna_trials.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")
log("optuna terminado")
trials.head(10)

,number,value,params_min_sum_hessian_in_leaf,params_num_leaves,user_attrs_best_iter,user_attrs_desvio_M,user_attrs_mejor_n,state
34,34,422.5210,9.553256,65,1427,2.906197,13348,COMPLETE
24,24,421.7895,2.344906,250,978,2.369817,13014,COMPLETE
21,21,420.6785,5.046243,38,1660,2.799797,13271,COMPLETE
36,36,420.1945,4.026507,106,1248,1.778585,14000,COMPLETE
16,16,419.9305,6.982303,64,1420,0.905982,13906,COMPLETE
18,18,419.7215,7.044084,47,1525,4.032277,13729,COMPLETE
15,15,419.6610,9.654357,42,1532,4.431772,13300,COMPLETE
32,32,418.5555,5.459563,53,1443,4.727998,13236,COMPLETE
37,37,416.7295,4.601432,101,1186,1.165870,13582,COMPLETE
17,17,416.2565,8.456115,148,1139,2.265613,13247,COMPLETE


In [4]:
# --- 4. Revalidar los 10 mejores con 10 semillas en junio y mayo (retoma lo guardado) ---
top = trials.head(TOP_REVALIDAR)
for corte in CORTES:
    tr, va, gan = datos_corte(corte)
    datos = lgb.Dataset(df.loc[tr, variables], label=df.loc[tr, "y_juntar"])
    X_va = df.loc[va, variables]
    for _, t in top.iterrows():
        p = {**params_fijos, "num_leaves": int(t["params_num_leaves"]),
             "min_sum_hessian_in_leaf": float(t["params_min_sum_hessian_in_leaf"])}
        for s in SEMILLAS:
            archivo = f"{CARPETA}/t{int(t['number'])}_{corte}_{s}.npy"
            if os.path.exists(archivo):
                continue
            inicio = time.time()
            modelo = lgb.train({**p, "seed": s}, datos, num_boost_round=int(t["user_attrs_best_iter"]))
            np.save(archivo, modelo.predict(X_va).astype("float32"))
            log(f"reval {corte} trial {int(t['number'])} semilla {s}: {time.time() - inicio:.0f} s")
    del datos, X_va
log("revalidación terminada")

In [5]:
# --- 5. Comparar contra la referencia (trial 34 de z005 con juntar) y aplicar la regla ---
filas = []
for corte in CORTES:
    tr, va, gan = datos_corte(corte)
    n_fijo = round(PROP_N * va.sum())
    fuentes = {"ref_t34": lambda s: f"{CARPETA_REF}/ref_{corte}_{s}.npy"}
    fuentes.update({int(n): (lambda s, n=int(n): f"{CARPETA}/t{n}_{corte}_{s}.npy") for n in top["number"]})
    for trial_id, ruta in fuentes.items():
        for s in SEMILLAS:
            acum = curva(np.load(ruta(s)), gan)
            filas.append({"corte": corte, "trial": trial_id, "semilla": s, "gan_max": acum.max() / 1e6,
                          "mejor_n": acum.argmax() + 1, "gan_nfijo": acum[n_fijo - 1] / 1e6, "gan_12000": acum[11_999] / 1e6})
por_semilla = pd.DataFrame(filas)
por_semilla.to_csv("z028_por_semilla.csv", sep=";", decimal=",", index=False, encoding="utf-8-sig")

idx = por_semilla.set_index(["trial", "corte", "semilla"]).sort_index()
res = []
for trial_id in [int(n) for n in top["number"]]:
    fila = {"trial": trial_id}
    for corte in CORTES:
        g, r = idx.loc[(trial_id, corte)], idx.loc[("ref_t34", corte)]
        d_max, d_fijo = g["gan_max"] - r["gan_max"], g["gan_nfijo"] - r["gan_nfijo"]
        fila.update({f"gan_max_{corte}": g["gan_max"].mean(), f"dif_max_{corte}": d_max.mean(), f"gana_{corte}": (d_max > 0).sum(),
                     f"dif_nfijo_{corte}": d_fijo.mean(), f"mejor_n_{corte}": g["mejor_n"].mean()})
    fila["pasa"] = all(fila[f"gana_{c}"] >= 8 and fila[f"dif_nfijo_{c}"] > 0 for c in CORTES)
    fila["promedio_dos_cortes"] = (fila["gan_max_junio"] + fila["gan_max_mayo"]) / 2
    res.append(fila)
res = pd.DataFrame(res).set_index("trial").join(
    top.set_index("number")[["value", "params_num_leaves", "params_min_sum_hessian_in_leaf", "user_attrs_best_iter"]]
       .rename(columns={"value": "optuna_5semillas"}))
res = res.sort_values("promedio_dos_cortes", ascending=False)
res.to_csv("z028_revalidacion.csv", sep=";", decimal=",", encoding="utf-8-sig", index_label="trial")
ref_prom = por_semilla[por_semilla.trial == "ref_t34"].groupby("corte")[["gan_max", "gan_nfijo"]].mean()
print("Referencia (trial 34 juntar), promedio 10 semillas:\n", ref_prom.round(1))
pasan = res[res["pasa"]].head(10)
print(f"\nPasan la regla: {len(pasan)} trials -> {list(pasan.index)}")
print("ADOPTAR ensamble nuevo" if len(pasan) >= 3 else "NO se adopta: el 32 queda como está")
log(f"pasan {len(pasan)}: {list(pasan.index)}")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
res.round(2)

Referencia (trial 34 juntar), promedio 10 semillas:
        gan_max  gan_nfijo
corte                    
junio    418.8      387.5
mayo     274.5      250.6

Pasan la regla: 0 trials -> []
NO se adopta: el 32 queda como está


,gan_max_junio,dif_max_junio,gana_junio,dif_nfijo_junio,mejor_n_junio,gan_max_mayo,dif_max_mayo,gana_mayo,dif_nfijo_mayo,mejor_n_mayo,pasa,promedio_dos_cortes,optuna_5semillas,params_num_leaves,params_min_sum_hessian_in_leaf,user_attrs_best_iter
trial,,,,,,,,,,,,,,,,
16,419.23,0.40,6,25.63,12527.3,278.95,4.48,10,5.39,8060.5,False,349.09,419.93,64,6.98,1420
18,418.03,-0.79,3,22.33,13518.8,279.93,5.46,10,7.15,7760.9,False,348.98,419.72,47,7.04,1525
34,416.61,-2.22,4,24.86,12726.6,281.17,6.70,10,3.41,7471.8,False,348.89,422.52,65,9.55,1427
15,417.79,-1.04,4,20.79,13591.7,279.97,5.51,9,6.16,7643.1,False,348.88,419.66,42,9.65,1532
21,417.44,-1.39,5,21.78,13644.5,280.28,5.81,10,6.38,8148.1,False,348.86,420.68,38,5.05,1660
37,417.87,-0.96,4,23.32,12904.7,278.74,4.27,10,5.39,8480.0,False,348.31,416.73,101,4.60,1186
36,418.92,0.10,4,22.11,13398.4,277.63,3.16,9,5.39,8648.5,False,348.28,420.19,106,4.03,1248
24,422.14,3.31,7,24.97,12841.4,274.28,-0.19,6,1.21,8810.2,False,348.21,421.79,250,2.34,978
32,415.27,-3.56,1,17.49,13775.4,280.90,6.43,9,4.95,8037.6,False,348.08,418.56,53,5.46,1443


In [6]:
# --- 6. Anotar en resultados_experimentos.csv ---
ruta = "resultados_experimentos.csv"
hist = pd.read_csv(ruta, sep=";", decimal=",", encoding="utf-8-sig")
hist = hist[~hist["descripcion"].str.startswith("z028")]
nuevas_filas = []
for trial_id, r in res.iterrows():
    for corte, (mes_train, mes_valid) in CORTES.items():
        nuevas_filas.append({
            "fecha": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
            "descripcion": (f"z028 Optuna juntar trial {trial_id} ({corte}): hojas {int(r['params_num_leaves'])}, "
                            f"hessian {r['params_min_sum_hessian_in_leaf']:.4f}, árboles {int(r['user_attrs_best_iter'])}; "
                            f"vs t34 juntar: dif máx {r[f'dif_max_{corte}']:+.1f} M, gana {int(r[f'gana_{corte}'])}/10, "
                            f"dif N fijo {r[f'dif_nfijo_{corte}']:+.1f} M; pasa: {'SÍ' if r['pasa'] else 'no'}"),
            "meses_train": str([mes_train]), "mes_valid": mes_valid, "semilla": str(SEMILLAS), "params": str(params_fijos),
            "mejor_n": int(r[f"mejor_n_{corte}"]), "ganancia_max_M": round(r[f"gan_max_{corte}"], 1)})
hist = pd.concat([hist, pd.DataFrame(nuevas_filas)], ignore_index=True)
hist["n_enviados"] = hist["n_enviados"].astype("Int64")
hist.to_csv(ruta, sep=";", decimal=",", index=False, encoding="utf-8-sig")
log("todo listo")

## Conclusiones (11/10, 02:05)

Corrida: Optuna 40 trials × 5 semillas (3 h 27 min) + revalidación 200 modelos (1 h 50 min).
Antes se lanzó una versión con 1 semilla por trial; se cortó en el trial 8 para pasar a 5 semillas (log en `z028_progreso_1semilla_cortado.txt`).
Windows (Control de aplicaciones) bloquea los .pyd de sqlalchemy: se corrió con una copia de sqlalchemy sin .pyd en `PYTHONPATH`.

**Regla: pasan 0 de 10 → por la regla, el 32/38 queda como está.**
- Falla solo en **ganancia máxima en junio**: empate (−5 a +3 M, ganan 1-7/10). La referencia llega a su máximo en N ≈ 14.800.
- **En N fijo (≈ 12.000, el N que se manda) ganan TODOS: junio +17,5 a +25,6 M en 10/10 semillas; mayo +1 a +7 M en 7-9/10.**
  En mayo también ganan la ganancia máx (+3 a +7 M, 9-10/10, salvo el 24).
- Los nuevos usan ~1.000-1.650 árboles (la referencia, 538): con juntar (el doble de positivos) conviene más árboles;
  ordenan mejor el top 12.000 aunque el máximo de la curva empate.
- Referencia (trial 34 z005 juntar, 10 semillas): junio 418,8 M máx / 387,5 N fijo; mayo 274,5 / 250,6.
- Candidatos si se decide igual un submit nuevo (N fijo ≥ 8/10 en los dos cortes y ganancia máx mayo ≥ 8/10):
  trials 16, 18, 34, 15, 21, 36, 32. **Decisión: Keila.**
